##### 19_endpoint_testing

##### 1. Notebook purpose


##### Objective

Validate the deployed Product Defect classifier through the actual
Databricks Model Serving endpoint.

Unlike local model evaluation, this notebook tests the complete deployed
system:

``` text

    image file
        ↓
    Base64 encoding
        ↓
    Databricks Serving Endpoint
        ↓
    deployed MLflow serving package
        ↓
    Base64 decoding
        ↓
    ResNet18 preprocessing
        ↓
    trained ResNet18 classifier
        ↓
    softmax + class mapping
        ↓
    structured prediction response

```
##### Tests

This notebook validates:

1. Endpoint deployment status
2. Known defect image
3. Known OK image
4. Response schema
5. Probability consistency
6. Known false-negative defect images
7. Missing input column
8. Invalid Base64 input
9. Empty Base64 input
10. Batch inference behavior

The known false-negative tests are particularly important for
Responsible AI because successful deployment does not eliminate known
model limitations.

##### 2. Imports

In [0]:
import base64
from pathlib import Path

import pandas as pd

from databricks.sdk import WorkspaceClient

from src.project_config import (
    RAW_DATA_DIR,
    REGISTERED_MODEL_NAME,
    SERVING_ENDPOINT_NAME,
)

In [0]:
workspace_client = WorkspaceClient()

print(
    "Endpoint:",
    SERVING_ENDPOINT_NAME,
)

print(
    "Registered model:",
    REGISTERED_MODEL_NAME,
)

print(
    "Dataset:",
    RAW_DATA_DIR,
)

##### Endpoint testing boundary

This notebook should behave like a client application.

The client should not need to know how the deployed neural network is
implemented.

Therefore, endpoint testing does not directly load:

- PyTorch
- ResNet18
- model weights
- MLflow model artifacts
- local inference functions

The endpoint itself owns the deployed inference pipeline.

This is an important production distinction:

- Notebook 18 --> knows how the model is packaged
- Notebook 19 --> only knows the serving contract

##### 3. Define test-data paths

In [0]:
TEST_DATA_DIR = (
    Path(RAW_DATA_DIR)
    / "test"
)

DEFECT_TEST_DIR = (
    TEST_DATA_DIR
    / "def_front"
)

OK_TEST_DIR = (
    TEST_DATA_DIR
    / "ok_front"
)

print(
    "Defect test directory:",
    DEFECT_TEST_DIR,
)

print(
    "OK test directory:",
    OK_TEST_DIR,
)

##### 4. Verify endpoint deployment state

In [0]:
#Before testing predictions, confirm that the deployed service is available.
endpoint = (
    workspace_client
    .serving_endpoints
    .get(
        SERVING_ENDPOINT_NAME
    )
)

print(
    "Endpoint:",
    endpoint.name,
)

print(
    "Ready:",
    endpoint.state.ready,
)

print(
    "Config update:",
    endpoint.state.config_update,
)

In [0]:
#Then add a fail-fast check:

if (
    str(endpoint.state.ready).upper()
    != "READY"
):
    raise RuntimeError(
        f"Endpoint '{SERVING_ENDPOINT_NAME}' "
        "is not READY."
    )

In [0]:
#Verify deployed model version

for entity in endpoint.config.served_entities:

    print(
        "Model:",
        entity.entity_name,
    )

    print(
        "Version:",
        entity.entity_version,
    )

    print(
        "Workload size:",
        entity.workload_size,
    )

##### 5. Create the client-side Base64 helper

In [0]:
def encode_image_base64(
    image_path: Path,
) -> str:
    """
    Encode an image file as a Base64 UTF-8 string
    for the serving endpoint.
    """

    if not image_path.exists():
        raise FileNotFoundError(
            f"Image not found: {image_path}"
        )

    with open(
        image_path,
        "rb",
    ) as image_file:

        return (
            base64.b64encode(
                image_file.read()
            )
            .decode("utf-8")
        )

Notice the separation:

- CLIENT --> encode image bytes → Base64
- SERVER --> Base64 → decode → preprocessing → model

The client does not resize, normalize, or convert the image into a tensor.

That belongs to the deployed inference pipeline.

##### 6. Create the endpoint invocation helper

In [0]:
#Now create one reusable helper for all tests.

def invoke_endpoint(
    model_input: pd.DataFrame,
):
    """
    Invoke the deployed Databricks serving endpoint.
    """

    response = (
        workspace_client
        .serving_endpoints
        .query(
            name=SERVING_ENDPOINT_NAME,
            dataframe_records=(
                model_input.to_dict(
                    orient="records"
                )
            ),
        )
    )

    return response

##### 7. TEST 1 — Known defect image

In [0]:
KNOWN_DEFECT_IMAGE = (
    DEFECT_TEST_DIR
    / "cast_def_0_1059.jpeg"
)

print(
    "Test image:",
    KNOWN_DEFECT_IMAGE,
)

print(
    "Expected class:",
    "def_front",
)

In [0]:
#encode
defect_base64 = (
    encode_image_base64(
        KNOWN_DEFECT_IMAGE
    )
)

print(
    "Base64 length:",
    len(defect_base64),
)

In [0]:
#Build the request:
defect_request = pd.DataFrame({
    "image_base64": [
        defect_base64
    ]
})

In [0]:
defect_response = (
    invoke_endpoint(
        defect_request
    )
)

print(
    defect_response
)

##### 8. Create a response parser

In [0]:
def parse_endpoint_response(
    response,
) -> pd.DataFrame:
    """
    Convert a Databricks serving endpoint response
    into a pandas DataFrame.
    """

    predictions = response.predictions

    if predictions is None:
        raise ValueError(
            "Endpoint response does not contain "
            "predictions."
        )

    if len(predictions) == 0:
        raise ValueError(
            "Endpoint returned an empty "
            "predictions list."
        )

    return pd.DataFrame(
        predictions
    )

In [0]:
defect_prediction = (
    parse_endpoint_response(
        defect_response
    )
)

display(
    defect_prediction
)

print(
    defect_prediction.columns.tolist()
)

##### 9. Finish TEST 1 — Known defect

In [0]:
EXPECTED_DEFECT_CLASS = (
    "def_front"
)

actual_defect_class = (
    defect_prediction.loc[
        0,
        "predicted_class",
    ]
)

assert (
    actual_defect_class
    ==
    EXPECTED_DEFECT_CLASS
), (
    "Known defect test failed. "
    f"Expected {EXPECTED_DEFECT_CLASS}, "
    f"received {actual_defect_class}."
)

print(
    "TEST 1 PASSED"
)

print(
    "Expected:",
    EXPECTED_DEFECT_CLASS,
)

print(
    "Predicted:",
    actual_defect_class,
)

print(
    "Confidence:",
    defect_prediction.loc[
        0,
        "confidence",
    ],
)

##### 10. TEST 2 — Known OK image

In [0]:
ok_images = sorted(
    OK_TEST_DIR.glob(
        "*.jpeg"
    )
)

if not ok_images:
    raise FileNotFoundError(
        "No OK test images found."
    )

KNOWN_OK_IMAGE = (
    ok_images[0]
)

print(
    "Test image:",
    KNOWN_OK_IMAGE,
)

print(
    "Expected class:",
    "ok_front",
)

In [0]:
ok_base64 = (
    encode_image_base64(
        KNOWN_OK_IMAGE
    )
)

ok_request = pd.DataFrame({
    "image_base64": [
        ok_base64
    ]
})

In [0]:
ok_response = (
    invoke_endpoint(
        ok_request
    )
)

ok_prediction = (
    parse_endpoint_response(
        ok_response
    )
)

display(
    ok_prediction
)

In [0]:
EXPECTED_OK_CLASS = (
    "ok_front"
)

actual_ok_class = (
    ok_prediction.loc[
        0,
        "predicted_class",
    ]
)

assert (
    actual_ok_class
    ==
    EXPECTED_OK_CLASS
), (
    "Known OK test failed. "
    f"Expected {EXPECTED_OK_CLASS}, "
    f"received {actual_ok_class}."
)

print(
    "TEST 2 PASSED"
)

print(
    "Expected:",
    EXPECTED_OK_CLASS,
)

print(
    "Predicted:",
    actual_ok_class,
)

print(
    "Confidence:",
    ok_prediction.loc[
        0,
        "confidence",
    ],
)

##### 11. TEST 3 — Response schema

In [0]:
#Now we're testing the API contract, not model accuracy.The serving package promised these output fields:

In [0]:
EXPECTED_RESPONSE_COLUMNS = {
    "predicted_class_index",
    "predicted_class",
    "confidence",
    "defect_probability",
    "ok_probability",
}

In [0]:
actual_response_columns = set(
    defect_prediction.columns
)

missing_columns = (
    EXPECTED_RESPONSE_COLUMNS
    -
    actual_response_columns
)

assert not missing_columns, (
    "Endpoint response is missing "
    f"columns: {missing_columns}"
)

print(
    "TEST 3 PASSED"
)

print(
    "Response schema:",
    sorted(
        actual_response_columns
    ),
)

In [0]:
actual_response_columns

In [0]:
missing_columns

##### 12. TEST 4 — Probability consistency

In [0]:
#For binary classification: P(defect) + P(OK) ≈ 1

def validate_probabilities(
    prediction: pd.DataFrame,
    tolerance: float = 1e-6,
) -> None:
    """
    Validate probability consistency for one
    endpoint prediction.
    """

    defect_probability = float(
        prediction.loc[
            0, # row0
            "defect_probability", # column
        ]
    )

    ok_probability = float(
        prediction.loc[
            0,
            "ok_probability",
        ]
    )

    probability_sum = (
        defect_probability
        +
        ok_probability
    )

    assert (
        abs(
            probability_sum - 1.0
        )
        <= tolerance
    ), (
        "Probabilities do not sum "
        f"to 1.0: {probability_sum}"
    )

    print(
        "Probability sum:",
        probability_sum,
    )

In [0]:
validate_probabilities(
    defect_prediction
)

validate_probabilities(
    ok_prediction
)

print(
    "TEST 4 PASSED"
)

In [0]:
def validate_confidence(
    prediction: pd.DataFrame,
    tolerance: float = 1e-6,
) -> None:

    predicted_class = (
        prediction.loc[
            0,
            "predicted_class",
        ]
    )

    confidence = float(
        prediction.loc[
            0,
            "confidence",
        ]
    )

    if predicted_class == "def_front":

        expected_confidence = float(
            prediction.loc[
                0,
                "defect_probability",
            ]
        )

    elif predicted_class == "ok_front":

        expected_confidence = float(
            prediction.loc[
                0,
                "ok_probability",
            ]
        )

    else:
        raise ValueError(
            "Unexpected predicted class: "
            f"{predicted_class}"
        )

    assert (
        abs(
            confidence
            -
            expected_confidence
        )
        <= tolerance
    ), (
        "Confidence is inconsistent "
        "with predicted-class probability."
    )

In [0]:
validate_confidence(
    defect_prediction
)

validate_confidence(
    ok_prediction
)

print(
    "Confidence values are consistent."
)

In [0]:
print(
    "defect_response.predictions:"
)

print(
    defect_response.predictions
)

print(
    "\ndefect_prediction:"
)

display(
    defect_prediction
)

print(
    "\ndefect_prediction columns:"
)

print(
    defect_prediction.columns.tolist()
)

##### 13. TEST 5 — Known false negatives

In [0]:
KNOWN_FALSE_NEGATIVE_FILES = [
    "cast_def_0_150.jpeg",
    "cast_def_0_1591.jpeg",
    "cast_def_0_2102.jpeg",
]

In [0]:
false_negative_results = []

for filename in (
    KNOWN_FALSE_NEGATIVE_FILES
):

    image_path = (
        DEFECT_TEST_DIR
        / filename
    )

    image_base64 = (
        encode_image_base64(
            image_path
        )
    )

    request = pd.DataFrame({
        "image_base64": [
            image_base64
        ]
    })

    response = (
        invoke_endpoint(
            request
        )
    )

    prediction = (
        parse_endpoint_response(
            response
        )
    )

    false_negative_results.append({
        "filename": filename,
        "actual_class": "def_front",
        "predicted_class": (
            prediction.loc[
                0,
                "predicted_class",
            ]
        ),
        "confidence": float(
            prediction.loc[
                0,
                "confidence",
            ]
        ),
        "defect_probability": float(
            prediction.loc[
                0,
                "defect_probability",
            ]
        ),
        "ok_probability": float(
            prediction.loc[
                0,
                "ok_probability",
            ]
        ),
    })

In [0]:
false_negative_df = (
    pd.DataFrame(
        false_negative_results
    )
)

display(
    false_negative_df
)

In [0]:
expected_known_behavior = (
    false_negative_df[
        "predicted_class"
    ]
    .eq("ok_front")
    .all()
)

assert expected_known_behavior, (
    "The deployed model no longer matches "
    "the previously evaluated behavior for "
    "the known false-negative cases."
)

print(
    "TEST 5 PASSED"
)

print(
    "All known false-negative cases "
    "remain reproducible through the "
    "deployed endpoint."
)

##### ⚖️ Responsible AI — False negatives after deployment

The endpoint successfully reproduces the three false-negative cases
identified during model evaluation.

This is a deployment-consistency success but a model-reliability
limitation.

In particular, a defect image can be classified as OK with high
confidence.

Therefore:

    high confidence ≠ guaranteed correctness

Deployment does not remove known model limitations.

For a real manufacturing quality-inspection system, the operational
impact of false negatives should be evaluated before determining
automation thresholds or human-review requirements.

##### 14. TEST 6 — Batch inference

In [0]:
batch_request = pd.DataFrame({
    "image_base64": [
        defect_base64,
        ok_base64,
    ]
})

In [0]:
batch_response = (
    invoke_endpoint(
        batch_request
    )
)

batch_predictions = (
    parse_endpoint_response(
        batch_response
    )
)

display(
    batch_predictions
)

In [0]:
assert (
    len(batch_predictions)
    ==
    len(batch_request)
), (
    "Batch response count does not "
    "match request count."
)

print(
    "TEST 6 PASSED"
)

print(
    "Batch size:",
    len(batch_request),
)

print(
    "Predictions returned:",
    len(batch_predictions),
)

##### failure-path endpoint testing

- Now we'll deliberately send bad requests. A production endpoint should not only handle valid requests correctly; it should also fail predictably and clearly for invalid ones.

- First, create a helper so expected errors don't make the notebook itself look broken:

In [0]:
def expect_endpoint_failure(
    test_name: str,
    model_input: pd.DataFrame,
) -> str:
    """
    Verify that an invalid endpoint request fails
    as expected.

    Returns the error message for inspection.
    """

    try:
        invoke_endpoint(
            model_input
        )

    except Exception as exc:
        error_message = str(exc)

        print(
            f"{test_name} PASSED"
        )

        print(
            "Endpoint rejected the request "
            "as expected."
        )

        print(
            "Error:"
        )

        print(
            error_message
        )

        return error_message

    raise AssertionError(
        f"{test_name} FAILED: "
        "endpoint unexpectedly accepted "
        "an invalid request."
    )

##### TEST 7 — Missing image_base64

In [0]:
missing_column_request = pd.DataFrame({
    "wrong_column": [
        "not_an_image"
    ]
})

missing_column_error = (
    expect_endpoint_failure(
        test_name=(
            "TEST 7 — Missing image_base64"
        ),
        model_input=(
            missing_column_request
        ),
    )
)

##### TEST 8 — Invalid Base64

In [0]:
invalid_base64_request = pd.DataFrame({
    "image_base64": [
        "this-is-not-valid-base64!!!"
    ]
})

invalid_base64_error = (
    expect_endpoint_failure(
        test_name=(
            "TEST 8 — Invalid Base64"
        ),
        model_input=(
            invalid_base64_request
        ),
    )
)

- TEST 7 --> API/schema problem
- TEST 8 --> content/data problem

##### TEST 9 — Empty Base64

In [0]:
empty_base64_request = pd.DataFrame({
    "image_base64": [
        ""
    ]
})

empty_base64_error = (
    expect_endpoint_failure(
        test_name=(
            "TEST 9 — Empty Base64"
        ),
        model_input=(
            empty_base64_request
        ),
    )
)

##### 15. Endpoint test summary

In [0]:
test_summary = pd.DataFrame([
    {
        "test": "Known defect image",
        "category": "Prediction",
        "result": "PASS",
        "expected_behavior": (
            "Predict def_front"
        ),
    },
    {
        "test": "Known OK image",
        "category": "Prediction",
        "result": "PASS",
        "expected_behavior": (
            "Predict ok_front"
        ),
    },
    {
        "test": "Response schema",
        "category": "Contract",
        "result": "PASS",
        "expected_behavior": (
            "Return all required fields"
        ),
    },
    {
        "test": "Probability consistency",
        "category": "Contract",
        "result": "PASS",
        "expected_behavior": (
            "Probabilities sum approximately to 1"
        ),
    },
    {
        "test": "Known false negatives",
        "category": "Model consistency",
        "result": "PASS",
        "expected_behavior": (
            "Reproduce evaluated model behavior"
        ),
    },
    {
        "test": "Batch inference",
        "category": "Serving",
        "result": "PASS",
        "expected_behavior": (
            "Return one prediction per image"
        ),
    },
    {
        "test": "Missing image_base64",
        "category": "Input validation",
        "result": "PASS",
        "expected_behavior": (
            "Reject missing required input"
        ),
    },
    {
        "test": "Invalid Base64",
        "category": "Input validation",
        "result": "PASS",
        "expected_behavior": (
            "Reject malformed Base64"
        ),
    },
    {
        "test": "Empty Base64",
        "category": "Input validation",
        "result": "PASS",
        "expected_behavior": (
            "Reject empty image input"
        ),
    },
])

display(
    test_summary
)

In [0]:
passed_tests = (
    test_summary[
        "result"
    ]
    .eq("PASS")
    .sum()
)

total_tests = len(
    test_summary
)

print(
    f"Tests passed: "
    f"{passed_tests}/{total_tests}"
)

assert (
    passed_tests
    ==
    total_tests
)

print(
    "ENDPOINT TEST SUITE PASSED"
)

> Passing the endpoint test suite means that the deployed serving 
> system behaves according to the tested serving contract.
>
> It does not mean that every model prediction is correct.
>
> The known false-negative test passes because the endpoint reproduces
> the previously evaluated model behavior, while those predictions
> remain model-quality errors.

##### 16. What exactly did we test?

##### What Did Endpoint Testing Validate?

Model evaluation and endpoint testing answer different questions.

##### Model evaluation

Model evaluation asks:

    How well does the classifier perform on labeled data?

Examples:

- accuracy
- precision
- recall
- F1
- confusion matrix
- false positives
- false negatives

##### Endpoint testing

Endpoint testing asks:

    Does the deployed system behave according to its serving contract?

This notebook validated:

- endpoint availability
- deployed model version
- Base64 request handling
- successful defect prediction
- successful OK prediction
- response schema
- probability consistency
- confidence consistency
- known model behavior
- multi-image batch requests
- missing-input rejection
- malformed-input rejection
- empty-input rejection

Therefore:

    model evaluation ≠ endpoint testing

Both are required for a production-oriented ML system.

##### 17. A subtle lesson from Test 7

##### MLflow Signature as a Runtime Contract

The missing-input test demonstrated why the MLflow model signature is
important.

The serving model was packaged with the required input:

    image_base64: string

When the client instead submitted:

    wrong_column

the serving system rejected the request because the required
`image_base64` field was missing.

The request therefore failed at the model-interface/schema boundary
before valid image inference could occur.

This demonstrates that an MLflow signature is not merely documentation.

In the serving system it also contributes to enforcement of the expected
model interface.

##### 18. 🔍 Explainable AI connection


Endpoint testing validates serving behavior, but it does not explain why
the neural network made a particular prediction.

Earlier Grad-CAM analysis investigated spatial regions that influenced
selected ResNet18 predictions.

The current endpoint returns:

- predicted class
- confidence
- defect probability
- OK probability

It does not return a Grad-CAM explanation.

Therefore:

    Endpoint testing
        → Is the deployed system behaving correctly?

    XAI
        → What image regions influenced a prediction?

These are complementary but different concerns.

The three known false-negative images remain particularly useful XAI
cases because the deployed system consistently reproduces those
incorrect predictions.

##### ⚖️ 19. Responsible AI (RAI)

Endpoint deployment and successful endpoint testing do not establish
that the classifier is safe for unrestricted production use.

##### 1. False-negative risk

Three known defect images are classified as OK.

In a manufacturing quality-inspection system, false negatives may be
particularly important because defective products could pass automated
inspection.

The operational consequence depends on the actual manufacturing use
case and must be assessed before production decisions are made.


##### 2. Confidence is not correctness

One known defect image is classified as OK with approximately 98.46%
confidence.

Therefore:

    high confidence ≠ guaranteed correctness

Model confidence should not be presented as a guarantee that a product
is defect-free.


##### 3. Evaluation-data integrity

Exact SHA-256 analysis previously identified overlap between the
development and original test data.

Evaluation on the non-duplicate test subset remained strong, but exact
hash analysis cannot rule out:

- near duplicates
- images from the same physical product
- sequential manufacturing images
- other correlated samples

Therefore, reported test performance should be interpreted with the
known dataset limitations.


##### 4. Distribution shift

The model has not been validated across all possible production
conditions, such as:

- different cameras
- different lighting
- different equipment
- different product batches
- changes in image positioning
- changes in manufacturing conditions

A READY endpoint does not establish robustness to these changes.


##### 5. Input validation

The endpoint correctly rejected:

- missing required input
- malformed Base64
- empty Base64

Rejecting malformed input is preferable to silently producing a
seemingly valid quality-inspection prediction.


##### 6. Human oversight

The appropriate role of human review cannot be determined from model
accuracy alone.

Operational decisions should consider:

- cost of false negatives
- cost of false positives
- inspection capacity
- safety requirements
- acceptable risk
- escalation procedures

A human-review threshold should therefore be based on actual operational
requirements rather than selected arbitrarily.

##### 20. Production testing architecture

``` text
                    CLIENT
                      │
                 JPEG image
                      │
                      ▼
               Base64 encoding
                      │
                      ▼
        ┌──────────────────────────┐
        │ Databricks Model Serving│
        └─────────────┬────────────┘
                      │
                      ▼
             MLflow input schema
                      │
          invalid ────┴────► reject
                      │
                    valid
                      ▼
          ProductDefectServingModel
                      │
                      ▼
             Base64 validation
                      │
          invalid ────┴────► reject
                      │
                    valid
                      ▼
                decode image
                      │
                      ▼
            ResNet18 preprocessing
                      │
                      ▼
             Fine-tuned ResNet18
                      │
                      ▼
                    logits
                      │
                      ▼
                   softmax
                      │
                      ▼
                 class mapping
                      │
                      ▼
              structured response
                      │
                      ▼
                 CLIENT TESTS
                      │
        ┌─────────────┼─────────────┐
        ▼             ▼             ▼
     Schema      Probabilities   Prediction

```

##### 21. Key Learnings

1. Endpoint testing validates the deployed system rather than only the
   underlying model.

2. A client application does not need to load PyTorch, ResNet18, or the
   trained model weights.

3. The client sends application-level input to the endpoint and depends
   on the serving contract.

4. Image preprocessing belongs inside the deployed inference pipeline,
   preventing clients from having to reproduce model preprocessing.

5. Base64 provides a convenient text representation for transporting
   image bytes through the serving interface.

6. MLflow signatures help define and enforce the expected model input
   contract.

7. Response-schema testing protects downstream applications from
   accidental serving-interface changes.

8. Probability tests should use numerical tolerances because floating-
   point calculations may not produce an exact mathematical value such
   as 1.0.

9. Batch inference should return one prediction for each supplied input.

10. Negative testing is an important part of production validation.

11. Missing, malformed, and empty inputs should fail clearly rather
    than silently producing predictions.

12. Endpoint consistency and model correctness are different concepts.

13. The deployed endpoint can correctly reproduce a previously known
    model error.

14. A passing endpoint test therefore does not imply that every
    prediction is correct.

15. High model confidence does not guarantee prediction correctness.

16. 🔍 XAI helps investigate why the model responded as it did.

17. ⚖️ RAI considers whether the resulting predictions can be
    appropriately trusted and used in the intended operational setting.

##### 22. Conclusion

The deployed Product Defect classifier has been validated through the
actual Databricks Model Serving endpoint.

The endpoint successfully:

- accepted Base64-encoded product images
- predicted a known defect image
- predicted a known OK image
- returned the documented response schema
- returned internally consistent probabilities and confidence
- supported multi-image batch inference
- reproduced the previously identified false-negative cases
- rejected requests missing the required input
- rejected malformed Base64
- rejected empty image input

The endpoint tests confirm that the deployed serving system behaves
consistently with the tested inference contract.

However, successful endpoint testing must not be confused with proof of
perfect model quality or production safety.

The three known false negatives remain model-quality limitations,
including one defect classified as OK with approximately 98.46%
confidence.

Together, model evaluation, XAI analysis, RAI assessment, deployment,
and endpoint testing provide a more complete understanding of the
system than model accuracy alone.

The Product Defect Computer Vision project now has a tested deployment
path from image input through a live Databricks serving endpoint.